# Practical 6: RNN and LSTM for Stock Price Prediction

**Aim:** Predict Reliance Industries (RELIANCE.NS) Open/High/Low/Close prices with a SimpleRNN and an LSTM and compare their RMSE.

In [ ]:
!pip install yfinance

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
import math

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ticker = "RELIANCE.NS"
data = yf.download(ticker, start="2015-01-01", end="2025-01-01")
data.columns = data.columns.get_level_values(0)
data = data[['Open', 'High', 'Low', 'Close']]

print(data.head())

In [ ]:
plt.figure()
plt.plot(data['Close'])
plt.title(f"{ticker} Closing Price")
plt.xlabel("Date")
plt.ylabel("Price")
plt.show()

In [ ]:
min_max_scaler = MinMaxScaler()
reliance_stock_norm = pd.DataFrame(
    min_max_scaler.fit_transform(data),
    columns=data.columns,
    index=data.index
)

In [ ]:
time_step = 60

X = []
y = []

for i in range(time_step, len(reliance_stock_norm)):
    X.append(reliance_stock_norm.iloc[i-time_step:i].values)
    y.append(reliance_stock_norm.iloc[i].values)

X = np.array(X)
y = np.array(y)

In [ ]:
split = int(len(X) * 0.8)

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense

rnn_model = Sequential()

rnn_model.add(SimpleRNN(64, input_shape=(X_train.shape[1], X_train.shape[2])))
rnn_model.add(Dense(4)) # 4 outputs (Open, High, Low, Close)

rnn_model.compile(optimizer='adam', loss='mean_squared_error')

rnn_model.summary()

In [ ]:
rnn_model.fit(X_train, y_train, epochs=20, batch_size=32)

In [ ]:
rnn_preds = rnn_model.predict(X_test)

rnn_preds_inv = min_max_scaler.inverse_transform(rnn_preds)
y_test_inv = min_max_scaler.inverse_transform(y_test)

In [ ]:
col_lst = list(reliance_stock_norm.columns)

figure, axes = plt.subplots(4, 1, figsize=(10, 15))

for i, col_name in enumerate(col_lst):

    rnn_preds_col = rnn_preds_inv[:, i]
    Ytest = y_test_inv[:, i]

    testScore = math.sqrt(mean_squared_error(Ytest, rnn_preds_col))
    print(col_name, ' Test Score: %.2f RMSE' % (testScore))

    data_col = data[col_name].values.reshape(-1,1)
    testPredictPlot = np.empty_like(data_col)
    testPredictPlot[:] = np.nan

    testPredictPlot[len(data_col)-len(rnn_preds_col):] = rnn_preds_col.reshape(-1,1)

    axes[i].plot(data_col, label="Actual")
    axes[i].plot(testPredictPlot, label="Predicted")
    axes[i].set_title(f"{col_name} RNN Prediction")
    axes[i].legend()

plt.tight_layout()
plt.show()

In [ ]:
lstm_model = Sequential()
lstm_model.add(LSTM(64, input_shape=(X_train.shape[1], X_train.shape[2])))
lstm_model.add(Dense(4))

lstm_model.compile(optimizer='adam', loss='mean_squared_error')

lstm_model.fit(X_train, y_train, epochs=20, batch_size=32)

In [ ]:
lstm_preds = lstm_model.predict(X_test)

lstm_preds_inv = min_max_scaler.inverse_transform(lstm_preds)
y_test_inv = min_max_scaler.inverse_transform(y_test)

In [ ]:
col_lst = list(reliance_stock_norm.columns)

figure, axes = plt.subplots(4, 1, figsize=(10, 15))

for i, col_name in enumerate(col_lst):

    lstm_preds_col = lstm_preds_inv[:, i]
    Ytest = y_test_inv[:, i]
    testScore = math.sqrt(mean_squared_error(Ytest, lstm_preds_col))
    print(col_name, ' Test Score: %.2f RMSE' % (testScore))

    # Create aligned plot
    data_col = data[col_name].values.reshape(-1,1)
    testPredictPlot = np.empty_like(data_col)
    testPredictPlot[:] = np.nan

    testPredictPlot[len(data_col)-len(lstm_preds_col):] = lstm_preds_col.reshape(-1,1)

    axes[i].plot(data_col, label="Actual")
    axes[i].plot(testPredictPlot, label="Predicted")
    axes[i].set_title(col_name + ' Prediction')
    axes[i].legend()

plt.tight_layout()
plt.show()